# SupplyVision - Data Transformation

## Module 4: ETL Implementation

This notebook implements the Transformation phase of the ETL pipeline.

The objective is to convert raw validated data into analytics-ready datasets by:

- Standardizing data types
- Optimizing memory usage
- Engineering new business features
- Creating a Date Dimension
- Exporting processed datasets

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

In [3]:
PROJECT_ROOT = Path("..")

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"

PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

In [4]:
tables = {}

for file in RAW_DATA_PATH.glob("*.csv"):
    table_name = file.stem
    tables[table_name] = pd.read_csv(file)

print(f"Loaded {len(tables)} tables.")

for table in sorted(tables):
    print(table)

Loaded 12 tables.
categories
customers
employees
order_items
orders
payments
products
promotions
returns
shipments
stores
suppliers


## Transformation Roadmap

The following transformations will be applied:

1. Convert date columns
2. Optimize numeric data types
3. Convert categorical columns
4. Create business features
5. Build Date Dimension
6. Export processed datasets

In [5]:
for table_name, df in tables.items():

    print("=" * 60)
    print(table_name.upper())
    print("=" * 60)

    print(df.dtypes)
    print()

CATEGORIES
category_id      int64
category_name      str
dtype: object

CUSTOMERS
customer_id    int64
city             str
signup_date      str
dtype: object

EMPLOYEES
employee_id    int64
store_id       int64
salary         int64
dtype: object

ORDERS
order_id        int64
customer_id     int64
store_id        int64
order_date        str
promotion_id    int64
dtype: object

ORDER_ITEMS
order_item_id    int64
order_id         int64
product_id       int64
qty              int64
price            int64
dtype: object

PAYMENTS
payment_id    int64
order_id      int64
amount        int64
dtype: object

PRODUCTS
product_id     int64
category_id    int64
supplier_id    int64
price          int64
dtype: object

PROMOTIONS
promotion_id    int64
discount        int64
dtype: object

RETURNS
return_id        int64
order_item_id    int64
refund           int64
dtype: object

SHIPMENTS
shipment_id    int64
order_id       int64
status           str
dtype: object

STORES
store_id    int64
city       

In [6]:
tables["customers"]["signup_date"] = pd.to_datetime(
    tables["customers"]["signup_date"]
)

tables["orders"]["order_date"] = pd.to_datetime(
    tables["orders"]["order_date"]
)

In [7]:
print(tables["customers"].dtypes)

print()

print(tables["orders"].dtypes)

customer_id             int64
city                      str
signup_date    datetime64[us]
dtype: object

order_id                 int64
customer_id              int64
store_id                 int64
order_date      datetime64[us]
promotion_id             int64
dtype: object


## Why Convert to Datetime?

Dates stored as strings cannot efficiently support:

- Time-based filtering
- Trend analysis
- Monthly reporting
- Quarterly reporting
- Time intelligence

Converting them to datetime enables powerful analytical operations in both Python and Power BI.

## Data Type Optimization

### Business Purpose

Raw datasets often use generic data types such as `int64` and `object`.

Optimizing data types reduces memory consumption and improves processing performance without changing the underlying data.

This is an important ETL optimization step, especially when working with large datasets.

In [8]:
def memory_usage_mb(df):
    return df.memory_usage(deep=True).sum() / 1024**2


memory_before = []

for table_name, df in tables.items():

    memory_before.append({
        "Table": table_name,
        "Memory (MB)": round(memory_usage_mb(df), 3)
    })

memory_before = pd.DataFrame(memory_before)

memory_before

,Table,Memory (MB)
0,categories,0.002
1,customers,3.386
2,employees,0.023
3,orders,11.444
4,order_items,22.888
5,payments,6.867
6,products,0.305
7,promotions,0.001
8,returns,0.687
9,shipments,20.504


In [9]:
for table_name, df in tables.items():

    int_columns = df.select_dtypes(include="int64").columns

    for col in int_columns:

        df[col] = pd.to_numeric(df[col], downcast="integer")

In [10]:
category_columns = {
    "customers": ["city"],
    "stores": ["city"],
    "suppliers": ["country"],
    "shipments": ["status"],
    "categories": ["category_name"]
}

for table_name, cols in category_columns.items():

    for col in cols:

        tables[table_name][col] = (
            tables[table_name][col]
            .astype("category")
        )

In [11]:
memory_after = []

for table_name, df in tables.items():

    memory_after.append({
        "Table": table_name,
        "Memory (MB)": round(memory_usage_mb(df), 3)
    })

memory_after = pd.DataFrame(memory_after)

memory_after

,Table,Memory (MB)
0,categories,0.002
1,customers,0.620
2,employees,0.007
3,orders,5.150
4,order_items,7.439
5,payments,2.861
6,products,0.067
7,promotions,0.000
8,returns,0.229
9,shipments,2.575


## Feature Engineering

### Business Purpose

Feature engineering creates new attributes from existing data to improve analytical capabilities.

Instead of repeatedly calculating year, month, quarter, and weekday during analysis, these attributes are created once during the ETL process and reused throughout the project.

In [12]:
orders = tables["orders"]

orders["order_year"] = orders["order_date"].dt.year

orders["order_quarter"] = "Q" + orders["order_date"].dt.quarter.astype(str)

orders["order_month"] = orders["order_date"].dt.month

orders["order_month_name"] = orders["order_date"].dt.month_name()

orders["order_day"] = orders["order_date"].dt.day

orders["order_day_name"] = orders["order_date"].dt.day_name()

orders["is_weekend"] = orders["order_date"].dt.dayofweek >= 5

orders.head()

,order_id,customer_id,store_id,order_date,promotion_id,order_year,order_quarter,order_month,order_month_name,order_day,order_day_name,is_weekend
0,1,45308,33,2021-08-26,24,2021,Q3,8,August,26,Thursday,False
1,2,10070,81,2022-03-19,3,2022,Q1,3,March,19,Saturday,True
2,3,43308,17,2021-01-21,25,2021,Q1,1,January,21,Thursday,False
3,4,47997,85,2021-01-16,48,2021,Q1,1,January,16,Saturday,True
4,5,36546,81,2022-09-14,33,2022,Q3,9,September,14,Wednesday,False


In [13]:
customers = tables["customers"]

customers["signup_year"] = customers["signup_date"].dt.year

customers["signup_month"] = customers["signup_date"].dt.month

customers["signup_month_name"] = customers["signup_date"].dt.month_name()

customers["signup_quarter"] = (
    "Q" + customers["signup_date"].dt.quarter.astype(str)
)

customers.head()

,customer_id,city,signup_date,signup_year,signup_month,signup_month_name,signup_quarter
0,1,Mumbai,2021-02-16,2021,2,February,Q1
1,2,Bangalore,2019-06-22,2019,6,June,Q2
2,3,Pune,2022-01-25,2022,1,January,Q1
3,4,Mumbai,2023-06-14,2023,6,June,Q2
4,5,Delhi,2023-06-24,2023,6,June,Q2
